In [71]:
import polars as pl

def classify_cancer_samples(df: pl.DataFrame) -> pl.DataFrame:
    """
    Classify samples as cancer / non-cancer / uncertain based on metadata text patterns.
    Returns the same DataFrame with added classification columns.
    """

    # --- Step 1: Setup ---
    PRIORITY_COLS = ["source_name", "tissue", "phenotype", "disease", "cell_type", "tumor_type"]
    PRIORITY_COLS = [c for c in PRIORITY_COLS if c in df.columns]

    # Regex patterns
    CANCER_POS = r"(?:\bcancers?\b|\btumou?rs?\b|\bmalignan(?:t|cy)\b|\bcarcinomas?\b|\bneoplasms?\b|\bmetasta(?:s|t)es?\b|\badenocarcinomas?\b|\bsarcomas?\b|\bleukemi(?:a|as)\b|\blymphom(?:a|as)\b|\bglioblastomas?\b|\bmelanomas?\b|\boncolog(?:y|ic|ical)\b)"
    CANCER_NEG = r"(?:\bnormal\b|\bhealthy\b|\bctrl\b|\badjacent normal\b|\bnon[-\s]?tumou?r(?:al)?\b|\bbenign\b|\bnon[-\s]?cancer(?:ous)?\b|\bsham\b|\bunaffected\b)"
    ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"

    # Helper to normalize text columns
    def normalize_text(col_expr):
        return (
            col_expr.cast(pl.Utf8)
            .fill_null("")
            .str.to_lowercase()
            .str.replace_all(r"[_/|]", " ")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
        )

    # --- Step 2: Sample name detection ---
    sample_name_col = "title" if "title" in df.columns else "biosample"

    df = df.with_columns([
        normalize_text(pl.col(sample_name_col)).str.contains(CANCER_POS).alias("cancer_in_sample_name"),
        normalize_text(pl.col(sample_name_col)).str.contains(CANCER_NEG).alias("negative_in_sample_name"),
        normalize_text(pl.col(sample_name_col)).str.contains(ONCO_TRAPS).alias("onco_trap_in_sample_name"),
    ])

    # --- Step 3: Check priority columns ---
    for col in PRIORITY_COLS:
        df = df.with_columns([
            normalize_text(pl.col(col)).str.contains(CANCER_POS).alias(f"cancer_in_{col}"),
            normalize_text(pl.col(col)).str.contains(CANCER_NEG).alias(f"negative_in_{col}"),
        ])

    # --- Step 4: Count mentions ---
    cancer_mention_cols = [f"cancer_in_{c}" for c in PRIORITY_COLS]
    negative_mention_cols = [f"negative_in_{c}" for c in PRIORITY_COLS]

    df = df.with_columns([
        pl.sum_horizontal([pl.col(c) for c in cancer_mention_cols if c in df.columns]).alias("n_cancer_mentions"),
        pl.sum_horizontal([pl.col(c) for c in negative_mention_cols if c in df.columns]).alias("n_negative_mentions"),
    ])

    # --- Step 5: Confidence category ---
    df = df.with_columns([
        pl.when(pl.col("onco_trap_in_sample_name"))
        .then(pl.lit("uncertain_onco_trap"))
        .when(
            pl.col("cancer_in_sample_name") &
            (pl.col("n_cancer_mentions") >= 1) &
            (pl.col("n_negative_mentions") == 0)
        )
        .then(pl.lit("confident_cancer"))
        .when(
            (pl.col("cancer_in_sample_name") & (pl.col("n_negative_mentions") == 0)) |
            (pl.col("n_cancer_mentions") >= 2)
        )
        .then(pl.lit("likely_cancer"))
        .when(
            pl.col("negative_in_sample_name") |
            (pl.col("n_negative_mentions") >= 1)
        )
        .then(pl.lit("likely_non_cancer"))
        .when(pl.col("n_cancer_mentions") == 1)
        .then(pl.lit("uncertain_weak_signal"))
        .otherwise(pl.lit("uncertain_no_signal"))
        .alias("confidence_category")
    ])

    return df


In [72]:
full_dataset = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)

df = pl.read_csv("data/train_test.csv")

In [73]:
df = df[["experiment_accession", "bioproject", "biosample", "sample_accession","run_accession", "is_cancer"]]

# idx = full_dataset.columns.index("cancer_type")

# full_dataset = full_dataset[full_dataset.columns[:idx+1]]

In [74]:
df.head()

# full_dataset.head()

experiment_accession,bioproject,biosample,sample_accession,run_accession,is_cancer
str,str,str,str,str,i64
"""SRX10489019""","""PRJNA718814""","""GSM5220969""","""SRS8615268""","""SRR14118670""",1
"""ERX1283444""","""PRJEB6455""","""SAMEA3724868""","""ERS1032017""","""ERR1211193""",0
"""SRX17708205""","""PRJNA884362""","""GSM6601021""","""SRS15238722""","""SRR21710838""",1
"""SRX4518392""","""PRJNA484951""","""GSM3323504""","""SRS3636903""","""SRR7655999""",0
"""SRX9364181""","""PRJNA671877""","""GSM4860401""","""SRS7586602""","""SRR12899131""",0


In [75]:
joined = full_dataset.join(
    df,
    on=["experiment_accession", "bioproject", "biosample", "sample_accession", "run_accession"],  # matching keys
    how="inner"  # keep only matching rows
)

In [76]:
predicted_df = classify_cancer_samples(joined)

In [77]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
cols_to_keep = predicted_df.columns[:idx + 1]

# add the two specific extra columns
cols_to_keep += ["is_cancer", "confidence_category"]

predicted_df.select(cols_to_keep)

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,cancer_type,is_cancer,confidence_category
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str
"""ena-EXPERIMENT-FUNCTIONAL GENO…","""ERX3526057""","""Illumina HiSeq 2500 paired end…",null,"""ERS3719163""","""2_Normal""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""Illumina HiSeq 2500""","""ERA2111341""","""2_Normal ena-SUBMISSION-FUNCTI…","""center""","""FUNCTIONAL GENOMICS CENTER ZUR…","""ERP117109""","""SAMEA5930682""","""PRJEB34234""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""",null,null,"""ERR3504843""","""13198053""","""3985812006""","""2403610007""","""TRUE""","""TRUE""","""1""","""968718068""","""1023813547""","""1112291056""","""878463736""","""2525599""","""carcinoma""",0,"""likely_non_cancer"""
"""GSM4860400""","""SRX9364180""","""GSM4860400: Fresh-frozen canin…",null,"""SRS7586601""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860400""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""Hovawart""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899130""","""32802407""","""3292101545""","""1101440073""","""TRUE""","""TRUE""","""1""","""798070483""","""761998966""","""845415170""","""879829853""","""6787073""","""tumor*""",0,"""likely_non_cancer"""
"""GSM4860401""","""SRX9364181""","""GSM4860401: Fresh-frozen canin…",null,"""SRS7586602""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860401""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""French Bulldog""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899131""","""36359099""","""3612160297""","""1220312604""","""TRUE""","""TRUE""","""1""","""889453430""","""826675433""","""915664165""","""965546260""","""14821009""","""tumor*""",0,"""likely_non_cancer"""
"""GSM3005111""","""SRX13999782""","""GSM3005111: D_Normal_Mucosa; C…",null,"""SRS11838164""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 4000""","""SRA1364912""","""GEO: GSE110661""","""center""","""NCBI""","""SRP357615""","""SAMN08550188""","""PRJNA434265""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""","""Bladder""",null,"""SRR17838664""","""9096025""","""2461885361""","""1015954149""","""TRUE""","""TRUE""",null,"""695834791""","""506960216""","""511045330""","""704969084""","""2648204""","""bladder cancer""",0,"""likely_non_cancer"""
"""GSM2498357""","""SRX2581899""","""GSM2498357: CHAD-P9; Canis lup…",null,"""SRS1995924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2000""","""SRA539403""","""nan GEO: GSE95183 nan nan GEO:…","""center""","""NCBI""","""SRP100514""","""GSM2498357""","""PRJNA376380""","""9615""","""Canis lupus familiaris""","""Portuguese Water Dog""",null,"""nan nan nan nan nan nan""","""nan spleen nan nan spleen nan""",null,"""SRR5278015""","""22714044""","""2271404400""","""1615616886""","""TRUE""","""TRUE""","""1""","""578763146""","""558428200""","""547283111""","""583938897""","""2991046""","""tumor*""",1,"""uncertain_no_signal"""
…,…,…,…,…,…

In [78]:
import spacy
import medspacy

# Option 1: Use medspacy pipeline directly
nlp = medspacy.load()

# Option 2: If you need to add to existing pipeline
# Load medspacy and copy the context component
negex = medspacy.load()
nlp.add_pipe("medspacy_context", source=negex, name="context", last=True)

In [79]:
text = "No evidence of metastatic cancer in the liver."

doc = nlp(text)
for ent in doc.ents:
    print(ent.text, ent._.is_negated, ent._.is_family, ent._.is_historical)


2025-10-28 01:39:48.693 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3357] [doc 0] Token 0 'No' marked as sentence start (span begin)
2025-10-28 01:39:48.693 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3357] Token/tag mapping: [(No, True), (evidence, False), (of, False), (metastatic, False), (cancer, False), (in, False), (the, False), (liver, False), (., False)]


In [80]:
import re

def clean_tissue(val: str) -> str:
    if not val or val.lower() == "nan":
        return None
    val = re.sub(r"\s+", " ", val.strip())
    tokens = [t for t in val.split() if t.lower() != "nan"]
    cleaned = []
    for t in tokens:
        if not cleaned or cleaned[-1].lower() != t.lower():
            cleaned.append(t)
    return " ".join(cleaned)


## First medspaCy

In [81]:
import spacy
import medspacy

# Load medspacy pipeline once (global scope for efficiency)
nlp = medspacy.load()


def analyze_text_with_spacy(text: str) -> dict:
    """
    Use spaCy/medspacy to detect cancer terms and check if they're negated.
    Returns a dict with: has_cancer_term, has_negated_cancer, has_confirmed_cancer
    """
    if not text or not isinstance(text, str) or not text.strip():
        return {
            "has_cancer_term": False,
            "has_negated_cancer": False,
            "has_confirmed_cancer": False
        }

    # Define cancer-related terms
    cancer_terms = [
        "cancer", "tumor", "tumour", "malignant", "malignancy",
        "carcinoma", "neoplasm", "metastatic", "metastasis",
        "adenocarcinoma", "sarcoma", "leukemia", "leukaemia",
        "lymphoma", "glioblastoma", "melanoma"
    ]

    doc = nlp(text.lower())

    has_cancer_term = False
    has_negated_cancer = False
    has_confirmed_cancer = False

    # Check entities detected by medspacy
    for ent in doc.ents:
        # Check if entity contains cancer-related terms
        if any(term in ent.text for term in cancer_terms):
            has_cancer_term = True

            # Check negation using medspacy's context detection
            if hasattr(ent._, "is_negated") and ent._.is_negated:
                has_negated_cancer = True
            else:
                has_confirmed_cancer = True

    # Fallback: if no entities but cancer terms present in text
    if not has_cancer_term and any(term in text.lower() for term in cancer_terms):
        # Run a simple token check
        tokens = text.lower().split()
        for term in cancer_terms:
            if term in tokens:
                has_cancer_term = True
                # Simple negation check for standalone terms
                idx = tokens.index(term)
                if idx > 0 and tokens[idx - 1] in ["no", "not", "without", "negative"]:
                    has_negated_cancer = True
                else:
                    has_confirmed_cancer = True
                break

    return {
        "has_cancer_term": has_cancer_term,
        "has_negated_cancer": has_negated_cancer,
        "has_confirmed_cancer": has_confirmed_cancer
    }


In [82]:
def classify_cancer_samples(df: pl.DataFrame) -> pl.DataFrame:
    """
    Classify samples as cancer / non-cancer / uncertain using spaCy NLP.
    Returns the same DataFrame with added classification columns.
    """

    # --- Step 1: Setup ---
    PRIORITY_COLS = ["source_name", "tissue", "phenotype", "disease", "cell_type", "tumor_type"]
    PRIORITY_COLS = [c for c in PRIORITY_COLS if c in df.columns]

    # Onco-traps pattern (still useful for false positives)
    ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"

    # Helper to normalize text columns
    def normalize_text(col_expr):
        return (
            col_expr.cast(pl.Utf8)
            .fill_null("")
            .str.to_lowercase()
            .str.replace_all(r"[_/|]", " ")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
        )
    # --- Step 1.5: Clean tissue column if present ---
    if "tissue" in df.columns:
        df = df.with_columns(
            pl.col("tissue").map_elements(clean_tissue, return_dtype=pl.Utf8)
        )

    # --- Step 2: Sample name detection using spaCy ---
    sample_name_col = "title" if "title" in df.columns else "biosample"

    # Extract texts directly from Polars (no pandas conversion)
    sample_texts = df.select(normalize_text(pl.col(sample_name_col)).alias("text"))["text"].to_list()

    # Analyze each sample name with spaCy
    spacy_results = [analyze_text_with_spacy(text) for text in sample_texts]

    df = df.with_columns([
        pl.Series("cancer_in_sample_name", [r["has_confirmed_cancer"] for r in spacy_results]),
        pl.Series("negative_in_sample_name", [r["has_negated_cancer"] for r in spacy_results]),
        normalize_text(pl.col(sample_name_col)).str.contains(ONCO_TRAPS).alias("onco_trap_in_sample_name"),
    ])

    # --- Step 3: Check priority columns with spaCy ---
    for col in PRIORITY_COLS:
        # Extract texts directly from Polars (no pandas conversion)
        col_texts = df.select(normalize_text(pl.col(col)).alias("text"))["text"].to_list()
        col_results = [analyze_text_with_spacy(text) for text in col_texts]

        df = df.with_columns([
            pl.Series(f"cancer_in_{col}", [r["has_confirmed_cancer"] for r in col_results]),
            pl.Series(f"negative_in_{col}", [r["has_negated_cancer"] for r in col_results]),
        ])

    # --- Step 4: Count mentions ---
    cancer_mention_cols = [f"cancer_in_{c}" for c in PRIORITY_COLS]
    negative_mention_cols = [f"negative_in_{c}" for c in PRIORITY_COLS]

    df = df.with_columns([
        pl.sum_horizontal([pl.col(c) for c in cancer_mention_cols if c in df.columns]).alias("n_cancer_mentions"),
        pl.sum_horizontal([pl.col(c) for c in negative_mention_cols if c in df.columns]).alias("n_negative_mentions"),
    ])

    # --- Step 5: Confidence category ---
    df = df.with_columns([
        pl.when(pl.col("onco_trap_in_sample_name"))
        .then(pl.lit("uncertain_onco_trap"))
        .when(
            pl.col("cancer_in_sample_name") &
            (pl.col("n_cancer_mentions") >= 1) &
            (pl.col("n_negative_mentions") == 0)
        )
        .then(pl.lit("confident_cancer"))
        .when(
            (pl.col("cancer_in_sample_name") & (pl.col("n_negative_mentions") == 0)) |
            (pl.col("n_cancer_mentions") >= 2)
        )
        .then(pl.lit("likely_cancer"))
        .when(
            pl.col("negative_in_sample_name") |
            (pl.col("n_negative_mentions") >= 1)
        )
        .then(pl.lit("likely_non_cancer"))
        .when(pl.col("n_cancer_mentions") == 1)
        .then(pl.lit("uncertain_weak_signal"))
        .otherwise(pl.lit("uncertain_no_signal"))
        .alias("confidence_category")
    ])

    # --- Step 6: Binary classification (1 = cancer, 0 = not cancer) ---
    df = df.with_columns([
        pl.when(pl.col("confidence_category").is_in(["confident_cancer", "likely_cancer"]))
        .then(pl.lit(1))
        .otherwise(pl.lit(0))
        .alias("predicted_cancer_binary")
    ])

    return df

In [83]:
predicted_df = classify_cancer_samples(joined)

2025-10-28 01:39:52.116 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3358] [doc 0] Token 0 'illumina' marked as sentence start (span begin)
2025-10-28 01:39:52.116 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3358] Token/tag mapping: [(illumina, True), (hiseq, False), (2500, False), (paired, False), (end, False), (sequencing, False)]
2025-10-28 01:39:52.118 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3359] [doc 0] Token 0 'gsm4860400' marked as sentence start (span begin)
2025-10-28 01:39:52.119 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3359] Token/tag mapping: [(gsm4860400, True), (:, False), (fresh, False), (-, False), (frozen, False), (canine, False), (healthy, False), (brain, False), (tissue, False), (-, False), (dog, False), (no, False), (., False), (60459, False), (;, False), (canis, False), (lupus, False), (familiaris, False), (;, Fa

In [84]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
cols_to_keep = predicted_df.columns[:idx + 1]

# add the two specific extra columns
cols_to_keep += ["is_cancer", "confidence_category", "predicted_cancer_binary"]

predicted_df.select(cols_to_keep)

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,cancer_type,is_cancer,confidence_category,predicted_cancer_binary
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i32
"""ena-EXPERIMENT-FUNCTIONAL GENO…","""ERX3526057""","""Illumina HiSeq 2500 paired end…",null,"""ERS3719163""","""2_Normal""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""Illumina HiSeq 2500""","""ERA2111341""","""2_Normal ena-SUBMISSION-FUNCTI…","""center""","""FUNCTIONAL GENOMICS CENTER ZUR…","""ERP117109""","""SAMEA5930682""","""PRJEB34234""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""",null,null,"""ERR3504843""","""13198053""","""3985812006""","""2403610007""","""TRUE""","""TRUE""","""1""","""968718068""","""1023813547""","""1112291056""","""878463736""","""2525599""","""carcinoma""",0,"""uncertain_no_signal""",0
"""GSM4860400""","""SRX9364180""","""GSM4860400: Fresh-frozen canin…",null,"""SRS7586601""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860400""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""Hovawart""",null,"""nan nan nan nan nan nan""","""control""",null,"""SRR12899130""","""32802407""","""3292101545""","""1101440073""","""TRUE""","""TRUE""","""1""","""798070483""","""761998966""","""845415170""","""879829853""","""6787073""","""tumor*""",0,"""uncertain_no_signal""",0
"""GSM4860401""","""SRX9364181""","""GSM4860401: Fresh-frozen canin…",null,"""SRS7586602""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860401""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""French Bulldog""",null,"""nan nan nan nan nan nan""","""control""",null,"""SRR12899131""","""36359099""","""3612160297""","""1220312604""","""TRUE""","""TRUE""","""1""","""889453430""","""826675433""","""915664165""","""965546260""","""14821009""","""tumor*""",0,"""uncertain_no_signal""",0
"""GSM3005111""","""SRX13999782""","""GSM3005111: D_Normal_Mucosa; C…",null,"""SRS11838164""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 4000""","""SRA1364912""","""GEO: GSE110661""","""center""","""NCBI""","""SRP357615""","""SAMN08550188""","""PRJNA434265""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""","""Bladder""",null,"""SRR17838664""","""9096025""","""2461885361""","""1015954149""","""TRUE""","""TRUE""",null,"""695834791""","""506960216""","""511045330""","""704969084""","""2648204""","""bladder cancer""",0,"""uncertain_no_signal""",0
"""GSM2498357""","""SRX2581899""","""GSM2498357: CHAD-P9; Canis lup…",null,"""SRS1995924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2000""","""SRA539403""","""nan GEO: GSE95183 nan nan GEO:…","""center""","""NCBI""","""SRP100514""","""GSM2498357""","""PRJNA376380""","""9615""","""Canis lupus familiaris""","""Portuguese Water Dog""",null,"""nan nan nan nan nan nan""","""spleen""",null,"""SRR5278015""","""22714044""","""2271404400""","""1615616886""","""TRUE""","""TRUE""","""1""","""578763146""","""558428200""","""547283111""","""583938897""","""2991046""","""tumor*""",1,"""uncertain_no_signal""",0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [85]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Convert to numpy arrays
y_true = predicted_df["is_cancer"].to_numpy()
y_pred = predicted_df["predicted_cancer_binary"].to_numpy()

# --- Metrics ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# --- Full report ---
print("\nClassification Report:\n", classification_report(y_true, y_pred))


Accuracy:  0.558
Precision: 0.920
Recall:    0.390
F1-score:  0.548

Confusion Matrix:
 [[50  4]
 [72 46]]

Classification Report:
               precision    recall  f1-score   support

           0       0.41      0.93      0.57        54
           1       0.92      0.39      0.55       118

    accuracy                           0.56       172
   macro avg       0.66      0.66      0.56       172
weighted avg       0.76      0.56      0.55       172



In [47]:
predicted_df.filter((pl.col("predicted_cancer_binary") == 0) & (pl.col("is_cancer") == 1))[:, :100]


experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,…,filename,filename2,treatment,biomaterial_provider,cell_type,sample_name,ena_first_public,insdc_center_name,insdc_status,broker_name,common_name,geographic_location_country_and_or_sea_,organism_part,scientific_name,ena_last_update,external_id,insdc_center_alias,insdc_first_public,insdc_last_update,developmental_stage,fraction,individual,specimen_with_known_storage_state,birth_date,umc_id,sub_species,tissue_source,clinical_description,tumor_location,disease_state,dog_breed,source_location,sample_type,phenotype,stimulus,tumor_type,diagnosis
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""GSM2498357""","""SRX2581899""","""GSM2498357: CHAD-P9; Canis lup…",null,"""SRS1995924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2000""","""SRA539403""","""nan GEO: GSE95183 nan nan GEO:…","""center""","""NCBI""","""SRP100514""","""GSM2498357""","""PRJNA376380""","""9615""","""Canis lupus familiaris""","""Portuguese Water Dog""",null,"""nan nan nan nan nan nan""","""spleen""",null,"""SRR5278015""","""22714044""","""2271404400""","""1615616886""","""TRUE""","""TRUE""","""1""","""578763146""","""558428200""","""547283111""","""583938897""","""2991046""",…,null,null,null,null,"""nan nan nan nan nan nan nan na…","""nan nan nan nan nan nan""","""nan nan nan nan nan nan""",null,null,"""nan nan nan nan nan nan""",null,null,null,null,"""nan nan nan nan nan nan""",null,null,null,null,null,null,null,null,null,null,"""familiaris""",null,null,null,null,null,null,"""nan nan nan nan nan nan""",null,null,null,null
"""GSM3384215""","""SRX4671104""","""GSM3384215: CMT-121-normal; Ca…",null,"""SRS3765858""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2500""","""SRA771342""","""nan GEO: GSE119810 nan nan GEO…","""center""","""NCBI""","""SRP159466""","""SAMN10034253""","""PRJNA489087""","""9615""","""Canis lupus familiaris""","""Mixed""",null,"""female nan female female nan f…","""mammary normal tissue""",null,"""SRR7819770""","""36081775""","""7288518550""","""3348858994""","""TRUE""","""TRUE""","""1""","""1781724667""","""1858786691""","""1858004193""","""1787247679""","""2755320""",…,null,null,null,null,null,"""nan nan nan nan nan nan""","""nan nan nan nan nan nan""",null,null,null,null,null,null,null,"""nan nan nan nan nan nan""",null,null,null,null,null,null,null,null,null,null,"""familiaris""",null,null,null,null,null,null,null,null,null,null,"""Complex carcinoma"""
"""32510 RNA-seq of spontaneous c…","""SRX295047""","""32510 RNA-seq of spontaneous c…",null,"""SRS429267""","""RNA-seq_32510""","""RNA-Seq""","""TRANSCRIPTOMIC""","""unspecified""","""PAIRED""","""Illumina HiSeq 2000""","""SRA081978""","""nan Canine mammary cancer sequ…","""center""","""UNIVERSITY OF GEORGIA""","""SRP024250""","""SAMN02144219""","""PRJNA203086""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""","""""",null,"""SRR882092""","""47399753""","""4739975300""","""3222090928""","""TRUE""","""TRUE""","""1""","""762910162""","""789851239""","""769751531""","""773285705""","""1007263""",…,null,null,null,null,null,"""nan nan nan nan nan nan""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,nu

## Testing medspaCy (NLP based approach)

In [40]:
import spacy
import medspacy
from medspacy.ner import TargetRule
# Note: TargetMatcher is fetched via pipeline, not imported directly

nlp = medspacy.load()  # load medspacy pipeline

# Retrieve the target matcher pipe
target_matcher = nlp.get_pipe("medspacy_target_matcher")

# Define your list of cancer-related terms
cancer_terms = ["cancer", "tumor", "tumour", "carcinoma", "neoplasm", "metastasis"]

# Build target rules
target_rules = [
    TargetRule(term, category="CANCER")
    for term in cancer_terms
]

# Add the rules to the matcher
target_matcher.add(target_rules)

# Test it
text = "The biopsy showed a malignant carcinoma and bone metastasis but no benign tumor."
doc = nlp(text)
for ent in doc.ents:
    print(ent.text, ent.label_, ent._.target_rule)

2025-10-28 00:01:09.823 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1367] [doc 0] Token 0 'The' marked as sentence start (span begin)
2025-10-28 00:01:09.825 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1367] Token/tag mapping: [(The, True), (biopsy, False), (showed, False), (a, False), (malignant, False), (carcinoma, False), (and, False), (bone, False), (metastasis, False), (but, False), (no, False), (benign, False), (tumor, False), (., False)]


carcinoma CANCER TargetRule(literal="carcinoma", category="CANCER", pattern=None, attributes=None, on_match=None)
metastasis CANCER TargetRule(literal="metastasis", category="CANCER", pattern=None, attributes=None, on_match=None)
tumor CANCER TargetRule(literal="tumor", category="CANCER", pattern=None, attributes=None, on_match=None)


In [41]:
import medspacy
from medspacy.ner import TargetRule

# Load full medspaCy pipeline (includes context/negation detection)
nlp = medspacy.load()
target_matcher = nlp.get_pipe("medspacy_target_matcher")

# Add some cancer-related target rules
for term in ["cancer", "tumor", "carcinoma"]:
    target_matcher.add([TargetRule(literal=term, category="CANCER")])

# Test examples
texts = [
    "There is no cancer here.",
    "Patient diagnosed with malignant tumor.",
    "No evidence of carcinoma detected."
]

for text in texts:
    doc = nlp(text)
    print(f"\nText: {text}")
    for ent in doc.ents:
        print(f" - Entity: {ent.text!r}, Category: {ent.label_}, Negated: {ent._.is_negated}")


2025-10-28 00:48:55.490 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1368] [doc 0] Token 0 'There' marked as sentence start (span begin)
2025-10-28 00:48:55.490 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1368] Token/tag mapping: [(There, True), (is, False), (no, False), (cancer, False), (here, False), (., False)]
2025-10-28 00:48:55.493 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1369] [doc 0] Token 0 'Patient' marked as sentence start (span begin)
2025-10-28 00:48:55.493 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1369] Token/tag mapping: [(Patient, True), (diagnosed, False), (with, False), (malignant, False), (tumor, False), (., False)]
2025-10-28 00:48:55.494 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1370] [doc 0] Token 0 'No' marked as sentence start (span begin)
2025-10-28 00:48:55.495 | DEBUG    


Text: There is no cancer here.
 - Entity: 'cancer', Category: CANCER, Negated: True

Text: Patient diagnosed with malignant tumor.
 - Entity: 'tumor', Category: CANCER, Negated: False

Text: No evidence of carcinoma detected.
 - Entity: 'carcinoma', Category: CANCER, Negated: True


## Second medspaCy (Failed Attempt)

In [42]:
import medspacy
from medspacy.ner import TargetRule

# --- Load medspacy pipeline ---
nlp = medspacy.load()
target_matcher = nlp.get_pipe("medspacy_target_matcher")

# Add cancer-related terms
CANCER_TERMS = [
    "cancer", "tumor", "tumour", "malignant", "malignancy",
    "carcinoma", "neoplasm", "metastatic", "metastasis",
    "adenocarcinoma", "sarcoma", "leukemia", "leukaemia",
    "lymphoma", "glioblastoma", "melanoma"
]
target_rules = [TargetRule(term, category="CANCER") for term in CANCER_TERMS]
target_matcher.add(target_rules)


def analyze_text_with_spacy(text: str) -> dict:
    """
    Detect cancer terms and check if they're negated using medspacy's context model.
    """
    if not text or not isinstance(text, str) or not text.strip():
        return {
            "has_cancer_term": False,
            "has_negated_cancer": False,
            "has_confirmed_cancer": False
        }

    doc = nlp(text.lower())

    has_cancer_term = False
    has_negated_cancer = False
    has_confirmed_cancer = False

    for ent in doc.ents:
        if ent.label_ == "CANCER":      # detected by target matcher
            has_cancer_term = True
            if getattr(ent._, "is_negated", False):
                has_negated_cancer = True
            else:
                has_confirmed_cancer = True

    return {
        "has_cancer_term": has_cancer_term,
        "has_negated_cancer": has_negated_cancer,
        "has_confirmed_cancer": has_confirmed_cancer
    }

In [53]:
# Test medspacy entity detection on real metadata samples
import spacy
import medspacy

# Load medspacy
nlp = medspacy.load()

# Sample texts from different scenarios
test_texts = [
    "breast cancer tissue",
    "normal breast tissue",
    "no evidence of tumor",
    "tumor sample from lung",
    "cancer cell line",
    "adjacent normal tissue",
    "metastatic carcinoma",
    "healthy control sample",
    "glioblastoma multiforme",
    "non-tumor breast tissue"
]

print("=" * 80)
print("TESTING MEDSPACY ENTITY DETECTION")
print("=" * 80)

for text in test_texts:
    doc = nlp(text)
    entities = [(ent.text, ent.label_, getattr(ent._, 'is_negated', None)) for ent in doc.ents]

    print(f"\nText: '{text}'")
    if entities:
        print(f"  ✓ Entities found: {entities}")
    else:
        print(f"  ✗ NO ENTITIES DETECTED")

print("\n" + "=" * 80)


2025-10-28 01:16:42.206 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2681] [doc 0] Token 0 'breast' marked as sentence start (span begin)
2025-10-28 01:16:42.206 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2681] Token/tag mapping: [(breast, True), (cancer, False), (tissue, False)]
2025-10-28 01:16:42.207 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2682] [doc 0] Token 0 'normal' marked as sentence start (span begin)
2025-10-28 01:16:42.207 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2682] Token/tag mapping: [(normal, True), (breast, False), (tissue, False)]
2025-10-28 01:16:42.208 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2683] [doc 0] Token 0 'no' marked as sentence start (span begin)
2025-10-28 01:16:42.208 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2683] Token/tag

TESTING MEDSPACY ENTITY DETECTION

Text: 'breast cancer tissue'
  ✗ NO ENTITIES DETECTED

Text: 'normal breast tissue'
  ✗ NO ENTITIES DETECTED

Text: 'no evidence of tumor'
  ✗ NO ENTITIES DETECTED

Text: 'tumor sample from lung'
  ✗ NO ENTITIES DETECTED

Text: 'cancer cell line'
  ✗ NO ENTITIES DETECTED

Text: 'adjacent normal tissue'
  ✗ NO ENTITIES DETECTED

Text: 'metastatic carcinoma'
  ✗ NO ENTITIES DETECTED

Text: 'healthy control sample'
  ✗ NO ENTITIES DETECTED

Text: 'glioblastoma multiforme'
  ✗ NO ENTITIES DETECTED

Text: 'non-tumor breast tissue'
  ✗ NO ENTITIES DETECTED



In [54]:
import spacy
import medspacy
from medspacy.ner import TargetRule

# Load medspacy pipeline
nlp = medspacy.load()

# Get the target matcher component
target_matcher = nlp.get_pipe("medspacy_target_matcher")

# Define cancer-related terms to detect
cancer_rules = [
    # General cancer terms
    TargetRule("cancer", "PROBLEM"),
    TargetRule("cancers", "PROBLEM"),
    TargetRule("tumor", "PROBLEM"),
    TargetRule("tumour", "PROBLEM"),
    TargetRule("tumors", "PROBLEM"),
    TargetRule("tumours", "PROBLEM"),

    # Malignancy terms
    TargetRule("malignant", "PROBLEM"),
    TargetRule("malignancy", "PROBLEM"),

    # Specific cancer types
    TargetRule("carcinoma", "PROBLEM"),
    TargetRule("adenocarcinoma", "PROBLEM"),
    TargetRule("sarcoma", "PROBLEM"),
    TargetRule("melanoma", "PROBLEM"),
    TargetRule("glioma", "PROBLEM"),
    TargetRule("glioblastoma", "PROBLEM"),
    TargetRule("leukemia", "PROBLEM"),
    TargetRule("leukaemia", "PROBLEM"),
    TargetRule("lymphoma", "PROBLEM"),
    TargetRule("myeloma", "PROBLEM"),

    # Metastasis
    TargetRule("metastatic", "PROBLEM"),
    TargetRule("metastasis", "PROBLEM"),
    TargetRule("metastases", "PROBLEM"),

    # Neoplasm
    TargetRule("neoplasm", "PROBLEM"),
    TargetRule("neoplastic", "PROBLEM"),
]

# Add all rules to the target matcher
target_matcher.add(cancer_rules)

print(f"✓ Added {len(cancer_rules)} cancer-related target rules to medspacy")


✓ Added 23 cancer-related target rules to medspacy


In [55]:
# Re-test with TargetMatcher configured
test_texts = [
    "breast cancer tissue",
    "normal breast tissue",
    "no evidence of tumor",
    "tumor sample from lung",
    "cancer cell line",
    "adjacent normal tissue",
    "metastatic carcinoma",
    "healthy control sample",
    "glioblastoma multiforme",
    "non-tumor breast tissue"
]

print("=" * 80)
print("TESTING WITH TARGET MATCHER")
print("=" * 80)

for text in test_texts:
    doc = nlp(text)

    print(f"\nText: '{text}'")

    if doc.ents:
        for ent in doc.ents:
            negated = "✗ NEGATED" if ent._.is_negated else "✓ AFFIRMED"
            print(f"  Entity: '{ent.text}' | Label: {ent.label_} | {negated}")
    else:
        print(f"  ✗ NO ENTITIES DETECTED")

print("\n" + "=" * 80)

2025-10-28 01:18:21.419 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2691] [doc 0] Token 0 'breast' marked as sentence start (span begin)
2025-10-28 01:18:21.421 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2691] Token/tag mapping: [(breast, True), (cancer, False), (tissue, False)]
2025-10-28 01:18:21.423 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2692] [doc 0] Token 0 'normal' marked as sentence start (span begin)
2025-10-28 01:18:21.424 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2692] Token/tag mapping: [(normal, True), (breast, False), (tissue, False)]
2025-10-28 01:18:21.426 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2693] [doc 0] Token 0 'no' marked as sentence start (span begin)
2025-10-28 01:18:21.426 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2693] Token/tag

TESTING WITH TARGET MATCHER

Text: 'breast cancer tissue'
  Entity: 'cancer' | Label: PROBLEM | ✓ AFFIRMED

Text: 'normal breast tissue'
  ✗ NO ENTITIES DETECTED

Text: 'no evidence of tumor'
  Entity: 'tumor' | Label: PROBLEM | ✗ NEGATED

Text: 'tumor sample from lung'
  Entity: 'tumor' | Label: PROBLEM | ✓ AFFIRMED

Text: 'cancer cell line'
  Entity: 'cancer' | Label: PROBLEM | ✓ AFFIRMED

Text: 'adjacent normal tissue'
  ✗ NO ENTITIES DETECTED

Text: 'metastatic carcinoma'
  Entity: 'metastatic' | Label: PROBLEM | ✓ AFFIRMED
  Entity: 'carcinoma' | Label: PROBLEM | ✓ AFFIRMED

Text: 'healthy control sample'
  ✗ NO ENTITIES DETECTED

Text: 'glioblastoma multiforme'
  Entity: 'glioblastoma' | Label: PROBLEM | ✓ AFFIRMED

Text: 'non-tumor breast tissue'
  Entity: 'tumor' | Label: PROBLEM | ✓ AFFIRMED

